# Gradients and checks · Module 05
Python 3 and NumPy only. A numerical derivative checks a calculation at one point; it does not replace the derivation. You will keep the checker you build here for the rest of the course.


## Part 1: Sweep the step size
A forward difference has error proportional to h, and a central difference to h squared. Very small steps then fail for a different reason: roundoff. Predict where the error is smallest before you run this.


In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=False)
rng = np.random.default_rng(205)
x = 1.; exact = np.cos(x)
forward = lambda h: (np.sin(x + h) - np.sin(x)) / h
central = lambda h: (np.sin(x + h) - np.sin(x - h)) / (2 * h)
print('      h       forward error   central error')
errors = {}
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13]:
    errors[h] = abs(central(h) - exact)
    print(f'{h:9.0e}   {abs(forward(h) - exact):12.3e}   {errors[h]:12.3e}')
assert errors[1e-5] < errors[1e-1] and errors[1e-5] < errors[1e-13]


      h       forward error   central error
    1e-01      4.294e-02      9.001e-04
    1e-03      4.208e-04      9.005e-08
    1e-05      4.207e-06      1.114e-11
    1e-07      4.183e-08      1.943e-10
    1e-09      5.254e-08      2.970e-09
    1e-11      1.169e-06      1.169e-06
    1e-13      7.339e-04      1.788e-04


## Part 2: A reusable gradient checker
Give it a loss function, an analytic gradient and a point. It reports the largest relative disagreement. Test it on a correct gradient, then on a deliberately wrong one, because a check that cannot fail proves nothing.


In [2]:
def check_gradient(loss, gradient, theta, h=1e-5):
    theta = np.array(theta, dtype=float); analytic = gradient(theta); numeric = np.zeros_like(theta)
    for j in range(theta.size):
        step = np.zeros_like(theta); step[j] = h
        numeric[j] = (loss(theta + step) - loss(theta - step)) / (2 * h)
    scale = np.maximum(1e-8, np.abs(analytic) + np.abs(numeric))
    return np.max(np.abs(analytic - numeric) / scale)
loss = lambda t: t[0] ** 2 + 3 * t[0] * t[1] + t[1] ** 2
right = lambda t: np.array([2 * t[0] + 3 * t[1], 3 * t[0] + 2 * t[1]])
wrong = lambda t: np.array([2 * t[0] + 3 * t[1], 3 * t[0] + 3 * t[1]])
print('correct gradient, disagreement:', check_gradient(loss, right, [1., 2.]))
print('wrong gradient, disagreement:  ', check_gradient(loss, wrong, [1., 2.]))
assert check_gradient(loss, right, [1., 2.]) < 1e-7 < check_gradient(loss, wrong, [1., 2.])


correct gradient, disagreement: 6.447604355140101e-12
wrong gradient, disagreement:   0.12499999999365313


## Part 3: A scalar computational graph
Each node remembers its value, its parents and the local derivative to each parent. `backward` visits nodes in reverse order and multiplies along every path. When a value is used twice, the two paths add.


In [3]:
class Node:
    def __init__(self, value, parents=(), local=()):
        self.value, self.parents, self.local, self.grad = value, parents, local, 0.
    @staticmethod
    def wrap(other): return other if isinstance(other, Node) else Node(other)
    def __add__(self, other):
        other = Node.wrap(other); return Node(self.value + other.value, (self, other), (1., 1.))
    def __mul__(self, other):
        other = Node.wrap(other); return Node(self.value * other.value, (self, other), (other.value, self.value))
    __radd__ = __add__; __rmul__ = __mul__
    def __sub__(self, other): return self + (Node.wrap(other) * -1.)
    def square(self): return Node(self.value ** 2, (self,), (2 * self.value,))
    def backward(self):
        order, seen = [], set()
        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node.parents: visit(parent)
                order.append(node)
        visit(self); self.grad = 1.
        for node in reversed(order):
            for parent, local in zip(node.parents, node.local): parent.grad += local * node.grad
w, b = Node(3.), Node(1.); x, target = 2., 5.
loss = 0.5 * (w * x + b - target).square()
loss.backward()
print('loss:', loss.value, 'dL/dw:', w.grad, 'dL/db:', b.grad)
assert (loss.value, w.grad, b.grad) == (2., 4., 2.)
x, y = Node(1.), Node(2.)
shared = x * x + 3. * x * y
shared.backward()
print('f = x^2 + 3xy, value and gradient:', shared.value, x.grad, y.grad)
assert (x.grad, y.grad) == (8., 3.)


loss: 2.0 dL/dw: 4.0 dL/db: 2.0
f = x^2 + 3xy, value and gradient: 7.0 8.0 3.0


## Part 4: Jacobians multiply, and shapes tell you how
`f` maps 2 numbers to 3, and `g` maps 3 numbers to 1. The chain rule multiplies a 1x3 row by a 3x2 Jacobian. Check the product against the checker from Part 2.


In [4]:
def f(t): return np.array([t[0] * t[1], t[0] + t[1] ** 2, np.sin(t[0])])
def jacobian(t): return np.array([[t[1], t[0]], [1., 2 * t[1]], [np.cos(t[0]), 0.]])
g = lambda y: float(y @ y)
composite = lambda t: g(f(t))
chain = lambda t: 2 * f(t) @ jacobian(t)
point = np.array([.7, -1.2])
print('Jacobian shape:', jacobian(point).shape, ' gradient shape:', chain(point).shape)
assert check_gradient(composite, chain, point) < 1e-7
hessian = np.array([[2., 3.], [3., 2.]])
print('Hessian of x^2+3xy+y^2 has eigenvalues', np.linalg.eigvalsh(hessian), 'so the origin is a saddle.')
h = np.array([.1, -.05]); quadratic = lambda t: t[0] ** 2 + 3 * t[0] * t[1] + t[1] ** 2
t0 = np.array([1., 2.]); grad0 = right(t0)
taylor = quadratic(t0) + grad0 @ h + .5 * h @ hessian @ h
print('Taylor approximation:', taylor, ' true value:', quadratic(t0 + h))
assert np.isclose(taylor, quadratic(t0 + h))


Jacobian shape: (3, 2)  gradient shape: (2,)
Hessian of x^2+3xy+y^2 has eigenvalues [-1.  5.] so the origin is a saddle.
Taylor approximation: 11.4475  true value: 11.4475


## Your experiment
Add `exp` and `log` nodes to the graph, derive a two-stage expression of your own, and confirm it with `check_gradient`. Then explain, from the table in Part 1, why the checker has a useful range of step sizes rather than a best answer of "as small as possible". Submit your derivation and the verified utility.
